In [1]:
from pathlib import Path
import sys
import warnings

import joblib
import pandas as pd
from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
CLEANED_DATA_PATH = DATA_DIR / "cleaned_comments.csv"
X_TRAIN_PATH = DATA_DIR / "X_train.npz"
X_TEST_PATH = DATA_DIR / "X_test.npz"
Y_TRAIN_PATH = DATA_DIR / "y_train.csv"
Y_TEST_PATH = DATA_DIR / "y_test.csv"
VECTORIZER_PATH = DATA_DIR / "tfidf_vectorizer.joblib"
LABEL_ENCODER_PATH = DATA_DIR / "label_encoder.joblib"
LABEL_MAPPING_PATH = DATA_DIR / "label_mapping.csv"
REQUIRED_COLUMNS = ["CommentText", "clean_text", "Sentiment"]
RANDOM_STATE = 42
TEST_SIZE = 0.2

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

warnings.filterwarnings("ignore")

from utils import load_cleaned_data


In [2]:
df = load_cleaned_data(CLEANED_DATA_PATH)

missing_columns = set(REQUIRED_COLUMNS) - set(df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

df = df.copy()
df["CommentText"] = df["CommentText"].fillna("").astype(str).str.strip()
df["clean_text"] = df["clean_text"].fillna("").astype(str).str.strip()
df["Sentiment"] = df["Sentiment"].astype("string").str.strip()

before = len(df)
df = df.dropna(subset=["Sentiment"])
df = df[(df["CommentText"].str.len() > 0) & (df["clean_text"].str.len() > 0)].reset_index(drop=True)

print(f"Loaded cleaned data: {before:,} rows -> {len(df):,} usable rows")
print("Columns:", df.columns.tolist())
print("Sentiment distribution:")
print(df["Sentiment"].value_counts())
df.head()


Loaded cleaned data: 90,477 rows -> 90,477 usable rows
Columns: ['CommentText', 'Sentiment', 'Language', 'clean_text']
Sentiment distribution:
Sentiment
Negative    32401
Positive    29679
Neutral     28397
Name: count, dtype: Int64


,CommentText,Sentiment,Language,clean_text
0,Anyone know what movie this is?,Neutral,en,anyone know movie
1,The fact they're holding each other back while...,Positive,en,fact holding back equally aggressive
2,waiting next video will be?,Neutral,en,waiting next video
3,Thanks for the great video. I don't understand...,Neutral,en,thanks great video not understand db continues...
4,Good person helping good people. This is how i...,Positive,en,good person helping good people america except...


 ------------------------------------------------------------------
 1. Label encoding
 ------------------------------------------------------------------


In [3]:
le = LabelEncoder()
df["label"] = le.fit_transform(df["Sentiment"])

label_mapping = pd.DataFrame({
    "Sentiment": le.classes_,
    "label": le.transform(le.classes_),
})

print("Label mapping:")
print(label_mapping.to_string(index=False))


Label mapping:
Sentiment  label
 Negative      0
  Neutral      1
 Positive      2


 ------------------------------------------------------------------
 2. Train / test split (stratified)
 ------------------------------------------------------------------


In [4]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["clean_text"],
    df[["label", "Sentiment"]],
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df["label"],
)

print(f"Train size: {len(X_train_text):,}  Test size: {len(X_test_text):,}")
print("Train label distribution:")
print(y_train["Sentiment"].value_counts(normalize=True).round(3))
print("Test label distribution:")
print(y_test["Sentiment"].value_counts(normalize=True).round(3))


Train size: 72,381  Test size: 18,096
Train label distribution:
Sentiment
Negative    0.358
Positive    0.328
Neutral     0.314
Name: proportion, dtype: Float64
Test label distribution:
Sentiment
Negative    0.358
Positive    0.328
Neutral     0.314
Name: proportion, dtype: Float64


 ------------------------------------------------------------------
 3. TF-IDF vectorization
Raw comments keep punctuation, emoji, contractions, and short patterns that are useful for sentiment. To improve generalization, the vocabulary is intentionally smaller and uses a higher minimum document frequency so the model focuses on repeatable patterns instead of rare phrases.
 ------------------------------------------------------------------


In [5]:
vectorizer = ColumnTransformer(
    transformers=[
        (
            "word_tfidf",
            TfidfVectorizer(
                max_features=20_000,
                ngram_range=(1, 2),
                min_df=4,
                max_df=0.95,
                sublinear_tf=True,
                strip_accents="unicode",
            ),
            "CommentText",
        ),
        (
            "char_tfidf",
            TfidfVectorizer(
                max_features=20_000,
                analyzer="char_wb",
                ngram_range=(3, 5),
                min_df=4,
                sublinear_tf=True,
                strip_accents="unicode",
            ),
            "CommentText",
        ),
    ],
    sparse_threshold=1.0,
)

X_train = vectorizer.fit_transform(df.loc[X_train_text.index, ["CommentText"]])
X_test = vectorizer.transform(df.loc[X_test_text.index, ["CommentText"]])

print("TF-IDF matrix shape (train):", X_train.shape)
print("TF-IDF matrix shape (test):", X_test.shape)
word_features = len(vectorizer.named_transformers_["word_tfidf"].vocabulary_)
char_features = len(vectorizer.named_transformers_["char_tfidf"].vocabulary_)
print(f"Vocabulary size: {word_features + char_features:,} ({word_features:,} word + {char_features:,} char)")


TF-IDF matrix shape (train): (72381, 40000)
TF-IDF matrix shape (test): (18096, 40000)
Vocabulary size: 40,000 (20,000 word + 20,000 char)


In [6]:
sparse.save_npz(X_TRAIN_PATH, X_train)
sparse.save_npz(X_TEST_PATH, X_test)
y_train.reset_index(drop=True).to_csv(Y_TRAIN_PATH, index=False)
y_test.reset_index(drop=True).to_csv(Y_TEST_PATH, index=False)
label_mapping.to_csv(LABEL_MAPPING_PATH, index=False)

joblib.dump(vectorizer, VECTORIZER_PATH)
joblib.dump(le, LABEL_ENCODER_PATH)

print("Saved feature artifacts:")
for path in [X_TRAIN_PATH, X_TEST_PATH, Y_TRAIN_PATH, Y_TEST_PATH, VECTORIZER_PATH, LABEL_ENCODER_PATH, LABEL_MAPPING_PATH]:
    print("-", path)


Saved feature artifacts:
- /Users/tiyasharma/Desktop/TIYA/AI ML/Project /Comments Sentimental Analysis  copy/data/X_train.npz
- /Users/tiyasharma/Desktop/TIYA/AI ML/Project /Comments Sentimental Analysis  copy/data/X_test.npz
- /Users/tiyasharma/Desktop/TIYA/AI ML/Project /Comments Sentimental Analysis  copy/data/y_train.csv
- /Users/tiyasharma/Desktop/TIYA/AI ML/Project /Comments Sentimental Analysis  copy/data/y_test.csv
- /Users/tiyasharma/Desktop/TIYA/AI ML/Project /Comments Sentimental Analysis  copy/data/tfidf_vectorizer.joblib
- /Users/tiyasharma/Desktop/TIYA/AI ML/Project /Comments Sentimental Analysis  copy/data/label_encoder.joblib
- /Users/tiyasharma/Desktop/TIYA/AI ML/Project /Comments Sentimental Analysis  copy/data/label_mapping.csv
